# Sensitivity analysis 1: patient bootstrap confidence intervals

Estimate **95% pointwise percentile confidence intervals** for AUC and balanced accuracy for all 127 NEWS2 subsets in four setting–outcome experiments, using **2,000 bootstrap samples from the original dataset**.

## Method

Each replicate samples N original patients with replacement before cohort/outcome filtering. The same sampled patients supply both settings and outcomes. Cross-validation and inner-fold balanced-accuracy threshold selection are repeated within every replicate. Copies of the same patient remain in a single fold. Metrics are mean outer-fold AUC and BA, matching the original aggregation; thresholds retain the original tie-breaking rule with reference 5.

Intervals are the 2.5th and 97.5th percentiles of replicate estimates. They are pointwise intervals for each prespecified subset, not simultaneous intervals or selection-adjusted intervals for a winning subset. Threshold-selection optimism is controlled through outer evaluation; the bootstrap additionally varies patient sampling and fold assignments.

**Execution:** prepared for local execution; no study analyses have been run while creating this notebook. Repeating 10-fold evaluation for 2,000 × 4 × 127 combinations is computationally intensive. Per-replicate checkpoints support resumption without changing seeds. The helper reuses the original evaluator rather than a new metric implementation.

## Eligibility

Missing FiO2 or any GCS component remains missing rather than being converted to a normal/abnormal category. Both notebooks use the same rule. This explicitly handles missingness and may differ from the original notebook’s `np.where` behavior. GCS totals outside 3–15 are treated as missing and excluded; inspect the eligibility audit before running. Unique patient IDs are required. If a replicate cannot support both outcome classes in every grouped fold, execution stops with its ID; replicates are never silently discarded.

**Resampling size:** 2,000 means 2,000 separate datasets, not 2,000 patients in one dataset. Each contains N draws with replacement from the N original patients. Eligibility filtering occurs after sampling. Every subset obtains 2,000 AUC estimates and 2,000 BA estimates per setting/outcome; each estimate is the mean of its outer-fold metrics. The 95% interval uses their 2.5th and 97.5th percentiles. Datasets are generated in memory, while their results are saved as separate replicate CSV files.

## 1. Setup and configuration

In [5]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Supports starting Jupyter from the repository root or this notebook folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "functions" / "news2_scale_deconstruction.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the NEWS2 repository or sensitivity folder.")
SENSITIVITY_DIR = ROOT / "sensitivity analisis"
for location in (ROOT, SENSITIVITY_DIR):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))

from sensitivity_helpers import (
    FEATURES, OUTCOMES, load_original_data, prepare_cohort,
    assign_grouped_folds, run_patient_bootstrap, audit_gcs,
)

DATA_PATH = ROOT / "data" / "NEWs2.csv"
RANDOM_STATE = 42
N_OUTER_FOLDS = 10
N_JOBS = 1

N_BOOTSTRAPS = 2000
OUTPUT_DIR = ROOT / "data" / "outputs" / "sensitivity" / "bootstrap_2000_valid_gcs"


## 2. Load the original patient dataset

Resampling uses this original dataset, not a saved result table or previously bootstrapped dataset.

In [6]:
raw_data = load_original_data(DATA_PATH)
pd.DataFrame({"original_patients": [len(raw_data)], "bootstrap_replicates": [N_BOOTSTRAPS]})

,original_patients,bootstrap_replicates
0,18001,2000


## 3. Run or resume the bootstrap

The initial estimate and all replicates use the same grouped CV procedure. Progress counts completed replicates; each contains all four experiments and 127 subsets. Keep configuration and dataset unchanged when resuming. Use a new output directory for changed settings.

In [7]:
bootstrap_summary = run_patient_bootstrap(
    raw_data, OUTPUT_DIR,
    n_bootstraps=N_BOOTSTRAPS,
    n_splits=N_OUTER_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)
bootstrap_summary.head()

Bootstrap datasets:   2%|2         | 43/2000 [00:00<?, ?dataset/s]

KeyboardInterrupt: 

## 4. Full-score intervals

The complete CSV includes all 508 experiment–subset combinations. Displayed intervals are 95% percentile confidence intervals.

In [8]:
columns = ["cohort", "outcome", "n_features", "auc_estimate",
           "auc_ci95_low", "auc_ci95_high", "ba_estimate", "ba_ci95_low", "ba_ci95_high"]
bootstrap_summary.loc[bootstrap_summary["n_features"].eq(7), columns]

NameError: name 'bootstrap_summary' is not defined

## 5. Interval plots for all combinations

Each setting–outcome figure has separate AUC and BA panels. Subsets are grouped by parameter count. Every estimate and interval is available in the exported CSV; the number of simultaneous intervals does not imply multiplicity adjustment.

In [ ]:
for (cohort, outcome), group in bootstrap_summary.groupby(["cohort", "outcome"], sort=False):
    ordered = group.sort_values(["n_features", "subset_key"]).reset_index(drop=True)
    x = np.arange(len(ordered))
    colors = plt.get_cmap("turbo")((ordered["n_features"].to_numpy() - 1) / 6)
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, constrained_layout=True)
    for ax, prefix, label in zip(axes, ["auc", "ba"], ["AUC", "Balanced accuracy"]):
        ax.vlines(x, ordered[f"{prefix}_ci95_low"], ordered[f"{prefix}_ci95_high"],
                  colors=colors, linewidth=1, alpha=.7)
        ax.scatter(x, ordered[f"{prefix}_estimate"], c=colors, s=10, zorder=3)
        ax.axhline(.5, color="gray", linestyle="--", linewidth=.8)
        ax.set_ylabel(f"{label} (95% CI)")
        ax.set_ylim(0, 1)
        ax.spines[["top", "right"]].set_visible(False)
    axes[-1].set_xlabel("All 127 subsets, ordered by number of parameters and subset key")
    fig.suptitle(f"{cohort} | {outcome}: 2,000 patient bootstrap samples")
    stem = f"{cohort.lower()}_{outcome.lower()}_bootstrap_ci95"
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

## Outputs and interpretation

`bootstrap_95ci_all_subsets.csv` contains original grouped-CV estimates and 95% intervals. `bootstrap_0000.csv` through `bootstrap_1999.csv` retain replicate metrics. `bootstrap_settings.json` records data fingerprint and configuration. Intervals quantify uncertainty under this resampling and evaluation procedure; they do not establish significance of between-subset differences.

Run all cells locally after supplying the original dataset. No confidence limits or conclusions are available until execution completes.